## Executive Summary

**Purpose:** Hold the stator dimensions that the inductance calculations need, and say where each named conductor position sits in the slot.

**What it does:** `SlotGeometry` describes an open or semi-closed slot whose conductor body tapers anywhere between a parallel slot and a parallel tooth (taper *t* from 0 to 1). `StatorGeometry` adds the bore, air gap, stack, yoke and end-winding overhang, and derives the slot pitch, tooth width and outer radius. `SlotGeometry.regions` splits the conductor body into the positions of a `WindingPattern` layout (over-under layers of equal copper area, or side-by-side halves).

**Next steps:** air-gap inductance from the winding function, then position-dependent slot leakage and tooth-tip leakage, both from these regions.

## How It Fits Into the Motor Design Workflow

**Upstream:** `emachines.winding.WindingPattern` — which phase sits at which named position of which slot.

**Downstream:** the inductance parts (air gap, slot leakage, tooth tip, end winding), each a function of a `WindingPattern` and a `StatorGeometry`.

## Inputs

![Stator cross-section showing three of the Q slots with bore radius, air gap, slot pitch, tooth width, slot depth, yoke height and outer radius; side view with stack length and end-winding overhang.](images/stator_geometry.svg)

![Slot close-up: (a) open slot, (b) semi-closed slot with opening, tooth-tip transition and tapered conductor body, (c) the named conductor positions for over-under, side-by-side and hairpin layouts.](images/slot_geometry.svg)

| Symbol | Meaning | Where it is set |
|---|---|---|
| $Q$ | number of slots | `StatorGeometry.Q` |
| $r_{bore}$ | stator bore radius | `StatorGeometry.r_bore` |
| $g$ | air gap (mechanical, before Carter correction) | `StatorGeometry.g` |
| $l_{stk}$ | stack length | `StatorGeometry.l_stk` |
| $h_y$ | yoke height | `StatorGeometry.h_y` |
| $l_{ew}$ | end-winding overhang beyond each end of the stack | `StatorGeometry.l_ew` |
| $b_0$, $h_0$ | slot opening width and height | `SlotGeometry.b0`, `.h0` |
| $h_1$ | tooth-tip transition (wedge) height, from $b_0$ out to $b_s$ | `SlotGeometry.h1` |
| $b_s$ | slot width at the top of the conductor body | `SlotGeometry.bs` |
| $h_s$ | conductor body depth | `SlotGeometry.hs` |
| $t$ | taper: 0 parallel slot, 1 parallel tooth | `SlotGeometry.t` |
| $h_{sep}$ | separator between radial layers | `SlotGeometry.h_sep` |

Derived: slot pitch $\tau_s = 2\pi r_{bore}/Q$; depth $y$ is measured from the top of the conductor body, at radius $r_{body} = r_{bore} + h_0 + h_1$; slot depth $h_{slot} = h_0 + h_1 + h_s$; outer radius $R_{out} = r_{bore} + h_{slot} + h_y$.

An **open slot** is $b_0 = b_s$ and $h_0 = h_1 = 0$: the conductor body starts at the bore.

## Slot Taper

The width of the conductor body at depth $y$ is

$$b(y) = b_s + t\,\frac{2\pi y}{Q}, \qquad 0 \le y \le h_s .$$

With $t = 1$ the tooth width $w_t(y) = 2\pi(r_{body} + y)/Q - b(y) = 2\pi r_{body}/Q - b_s$ is the same at every depth, a parallel tooth. With $t = 0$ the slot sides are parallel. Values in between split the difference. Because $b(y)$ is linear in $y$, every slot integral in the leakage calculations stays in closed form.

The copper area between depths $y_0$ and $y_1$ is

$$A(y_0, y_1) = b_s (y_1 - y_0) + t\,\frac{\pi}{Q}\,(y_1^2 - y_0^2).$$

**Over-under layers** share the body between $n$ layers of equal copper area, with a separator $h_{sep}$ between neighbours. Each boundary solves $A(y_0, y) = A_{layer}$, a quadratic in $y$. Layer 1 (`top` when $n = 2$) is on the air-gap side. **Side-by-side** positions each take the full depth and half the width (`left` is toward slot $i - 1$).

In [ ]:
#| default_exp inductance.geometry

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass

import numpy as np

from emachines.winding.pattern import slot_positions

---

## `SlotRegion` and `SlotGeometry`

In [ ]:
#| export
@dataclass(frozen=True)
class SlotRegion:
    """
    The part of the conductor body that one named position occupies.

    Attributes
    ----------
    name   : str    Position name (see ``slot_positions``)
    y0, y1 : float  Depth range below the top of the conductor body [m], y0 < y1
    x0, x1 : float  Share of the local slot width, from the left side (0) to the right (1)
    """
    name: str
    y0: float
    y1: float
    x0: float = 0.0
    x1: float = 1.0


@dataclass(frozen=True)
class SlotGeometry:
    """
    Slot shape: an optional opening and tooth-tip transition above a tapered conductor body.

    All lengths in metres. Use ``SlotGeometry.open`` or ``SlotGeometry.semi_closed``.

    Attributes
    ----------
    bs    : float         Slot width at the top of the conductor body
    hs    : float         Conductor body depth
    b0    : float         Opening width; 0 -> bs (open slot)
    h0    : float         Opening height (0 for an open slot)
    h1    : float         Tooth-tip transition height, from b0 out to bs (0 for an open slot)
    t     : float         Taper, 0 = parallel slot ... 1 = parallel tooth
    h_sep : float         Separator height between radial layers (default 0)
    """
    bs: float
    hs: float
    b0: float = 0.0
    h0: float = 0.0
    h1: float = 0.0
    t: float = 0.0
    h_sep: float = 0.0

    def __post_init__(self):
        if self.b0 == 0:
            object.__setattr__(self, "b0", self.bs)
        if self.bs <= 0 or self.hs <= 0:
            raise ValueError(f"bs and hs must be positive, got bs={self.bs}, hs={self.hs}")
        if not 0 < self.b0 <= self.bs:
            raise ValueError(f"the opening b0 must be in (0, bs], got b0={self.b0}, bs={self.bs}")
        if self.h0 < 0 or self.h1 < 0 or self.h_sep < 0:
            raise ValueError("h0, h1 and h_sep must not be negative")
        if self.b0 < self.bs and self.h0 + self.h1 == 0:
            raise ValueError("a semi-closed slot (b0 < bs) needs h0 + h1 > 0")
        if not 0.0 <= self.t <= 1.0:
            raise ValueError(f"taper t must be between 0 (parallel slot) and 1 (parallel tooth), got t={self.t}")

    @classmethod
    def open(cls, bs: float, hs: float, t: float = 0.0, h_sep: float = 0.0) -> "SlotGeometry":
        """An open slot: the conductor body starts at the bore, opening width = bs."""
        return cls(bs=bs, hs=hs, t=t, h_sep=h_sep)

    @classmethod
    def semi_closed(cls, b0: float, h0: float, h1: float, bs: float, hs: float,
                    t: float = 0.0, h_sep: float = 0.0) -> "SlotGeometry":
        """A semi-closed slot with opening b0 × h0 and a tooth-tip transition of height h1."""
        return cls(bs=bs, hs=hs, b0=b0, h0=h0, h1=h1, t=t, h_sep=h_sep)

    @property
    def is_open(self) -> bool:
        """True if there is no opening or tooth tip (b0 = bs, h0 = h1 = 0)."""
        return self.b0 == self.bs and self.h0 == 0 and self.h1 == 0

    @property
    def depth(self) -> float:
        """Total slot depth h0 + h1 + hs."""
        return self.h0 + self.h1 + self.hs

    def width(self, y, Q: int):
        """Body width b(y) = bs + t·2πy/Q at depth y below the top of the body."""
        return self.bs + self.t * 2.0 * math.pi * np.asarray(y, dtype=float) / Q

    def bottom_width(self, Q: int) -> float:
        """Body width at the slot bottom, b(hs)."""
        return float(self.width(self.hs, Q))

    def area(self, Q: int, y0: float = 0.0, y1: float | None = None) -> float:
        """Body cross-section between depths y0 and y1 (default: the whole body) [m²]."""
        y1 = self.hs if y1 is None else y1
        return self.bs * (y1 - y0) + self.t * math.pi / Q * (y1 ** 2 - y0 ** 2)

    def _depth_for_area(self, Q: int, y0: float, a: float) -> float:
        """Depth y > y0 with area(Q, y0, y) = a."""
        c = self.t * math.pi / Q
        if c == 0:
            return y0 + a / self.bs
        k = a + self.bs * y0 + c * y0 ** 2
        return (-self.bs + math.sqrt(self.bs ** 2 + 4 * c * k)) / (2 * c)

    def regions(self, layout: str, Q: int, n: int = 2) -> tuple[SlotRegion, ...]:
        """
        Split the conductor body into the named positions of a layout.

        Parameters
        ----------
        layout : str  'single', 'over_under' or 'side_by_side'
        Q      : int  Slot count (sets the taper)
        n      : int  Radial layers for 'over_under' (default 2)

        Returns
        -------
        tuple of SlotRegion, in the order of ``slot_positions(layout, n)``.
        Over-under layers have equal copper area, separated by h_sep, layer 1 on the
        air-gap side. Side-by-side halves each take the full depth.
        """
        names = slot_positions(layout, n)
        if layout == "single":
            return (SlotRegion(names[0], 0.0, self.hs),)
        if layout == "side_by_side":
            return (SlotRegion(names[0], 0.0, self.hs, 0.0, 0.5), SlotRegion(names[1], 0.0, self.hs, 0.5, 1.0))
        k = len(names)
        copper = self.hs - (k - 1) * self.h_sep
        if copper <= 0:
            raise ValueError(f"{k - 1} separators of {self.h_sep} m leave no room in a body of depth {self.hs} m")
        # Each layer gets the same copper area. Where the separators sit depends on the
        # split, so start from equal heights and iterate (converges in a few passes).
        bounds: list[tuple[float, float]] = []
        a_layer = (self.area(Q) - sum(self.area(Q, (j + 1) * copper / k + j * self.h_sep,
                                                (j + 1) * copper / k + (j + 1) * self.h_sep)
                                      for j in range(k - 1))) / k
        for _ in range(50):
            y, bounds, a_sep_total = 0.0, [], 0.0
            for j in range(k):
                y1 = self._depth_for_area(Q, y, a_layer)
                bounds.append((y, y1))
                if j < k - 1:
                    a_sep_total += self.area(Q, y1, y1 + self.h_sep)
                    y = y1 + self.h_sep
            new = (self.area(Q) - a_sep_total) / k
            if abs(new - a_layer) <= 1e-14 * a_layer:
                break
            a_layer = new
        y_end = bounds[-1][1]
        bounds[-1] = (bounds[-1][0], self.hs)               # absorb rounding at the slot bottom
        assert abs(y_end - self.hs) < 1e-9 * max(1.0, self.hs)
        return tuple(SlotRegion(nm, a, b) for nm, (a, b) in zip(names, bounds))

In [ ]:
# Examples
s = SlotGeometry.open(bs=6e-3, hs=18e-3)
assert s.is_open and s.b0 == 6e-3 and s.depth == 18e-3
top, bottom = s.regions("over_under", Q=36)
assert np.isclose(top.y1, 9e-3) and np.isclose(bottom.y0, 9e-3)        # parallel slot: equal heights

sc = SlotGeometry.semi_closed(b0=2.5e-3, h0=1e-3, h1=1.5e-3, bs=6e-3, hs=18e-3, t=0.5)
print(f"bottom width {sc.bottom_width(36) * 1e3:.3f} mm, depth {sc.depth * 1e3:.1f} mm")
for r in sc.regions("over_under", Q=36):
    print(f"  {r.name:6s} y = {r.y0 * 1e3:6.3f} … {r.y1 * 1e3:6.3f} mm, area {sc.area(36, r.y0, r.y1) * 1e6:.3f} mm²")

---

## `StatorGeometry`

In [ ]:
#| export
@dataclass(frozen=True)
class StatorGeometry:
    """
    Stator dimensions for the inductance calculations (inner-rotor, radial flux). SI units.

    Attributes
    ----------
    Q      : int           Number of slots
    r_bore : float         Bore radius [m]
    g      : float         Mechanical air gap [m]
    l_stk  : float         Stack length [m]
    slot   : SlotGeometry  Slot shape
    h_y    : float         Yoke height [m]
    l_ew   : float         End-winding overhang beyond each end of the stack [m] (default 0)
    """
    Q: int
    r_bore: float
    g: float
    l_stk: float
    slot: SlotGeometry
    h_y: float
    l_ew: float = 0.0

    def __post_init__(self):
        if self.Q < 2:
            raise ValueError(f"Q must be >= 2, got Q={self.Q}")
        for name in ("r_bore", "g", "l_stk", "h_y"):
            if getattr(self, name) <= 0:
                raise ValueError(f"{name} must be positive, got {getattr(self, name)}")
        if self.l_ew < 0:
            raise ValueError(f"l_ew must not be negative, got {self.l_ew}")
        if self.slot.b0 >= self.tau_s:
            raise ValueError(f"slot opening b0 = {self.slot.b0} m leaves no tooth tip at slot pitch {self.tau_s:.6g} m")
        if min(self.tooth_width, self.tooth_width_bottom) <= 0:
            raise ValueError(f"slots too wide for {self.Q} slots on r_bore = {self.r_bore} m: "
                             f"tooth width {self.tooth_width:.6g} m at the top of the body, "
                             f"{self.tooth_width_bottom:.6g} m at the bottom")

    @property
    def tau_s(self) -> float:
        """Slot pitch at the bore, 2π r_bore / Q."""
        return 2.0 * math.pi * self.r_bore / self.Q

    @property
    def r_body(self) -> float:
        """Radius of the top of the conductor body, r_bore + h0 + h1."""
        return self.r_bore + self.slot.h0 + self.slot.h1

    @property
    def h_slot(self) -> float:
        """Total slot depth h0 + h1 + hs."""
        return self.slot.depth

    @property
    def r_slot_bottom(self) -> float:
        """Radius of the slot bottom."""
        return self.r_bore + self.h_slot

    @property
    def R_out(self) -> float:
        """Outer radius r_bore + h_slot + h_y."""
        return self.r_slot_bottom + self.h_y

    def slot_width(self, y):
        """Conductor-body width at depth y below the top of the body."""
        return self.slot.width(y, self.Q)

    def tooth_width_at(self, y):
        """Tooth width at depth y below the top of the conductor body."""
        return 2.0 * math.pi * (self.r_body + np.asarray(y, dtype=float)) / self.Q - self.slot.width(y, self.Q)

    @property
    def tooth_width(self) -> float:
        """Tooth width w_t at the top of the conductor body."""
        return float(self.tooth_width_at(0.0))

    @property
    def tooth_width_bottom(self) -> float:
        """Tooth width at the slot bottom."""
        return float(self.tooth_width_at(self.slot.hs))

    def regions(self, layout: str, n: int = 2) -> tuple[SlotRegion, ...]:
        """The named slot positions of a layout, as ``SlotGeometry.regions`` with this Q."""
        return self.slot.regions(layout, self.Q, n)

### Example

The machine drawn in the figures above: 36 slots on a 60 mm bore, 0.8 mm air gap, semi-closed slots tapered halfway between parallel slot and parallel tooth.

In [ ]:
stator = StatorGeometry(Q=36, r_bore=60e-3, g=0.8e-3, l_stk=100e-3, h_y=12e-3, l_ew=22e-3,
                        slot=SlotGeometry.semi_closed(b0=2.5e-3, h0=1e-3, h1=1.5e-3, bs=6e-3, hs=18e-3, t=0.5))
print(f"slot pitch {stator.tau_s * 1e3:.3f} mm, tooth {stator.tooth_width * 1e3:.3f} → {stator.tooth_width_bottom * 1e3:.3f} mm, "
      f"slot bottom width {stator.slot.bottom_width(36) * 1e3:.3f} mm, R_out {stator.R_out * 1e3:.1f} mm")
assert np.isclose(stator.R_out, 92.5e-3) and np.isclose(stator.tau_s, 2 * np.pi * 60e-3 / 36)

The same stator with the taper swept from parallel slot to parallel tooth. The tooth narrows toward the slot bottom at *t* = 0 and keeps its width at *t* = 1:

In [ ]:
import matplotlib.pyplot as plt
from dataclasses import replace

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.4))
y = np.linspace(0, 18e-3, 50)
for t in (0.0, 0.5, 1.0):
    st = replace(stator, slot=replace(stator.slot, t=t))
    ax.plot(y * 1e3, st.slot_width(y) * 1e3, label=f"slot, t = {t:g}")
    ax.plot(y * 1e3, st.tooth_width_at(y) * 1e3, "--", label=f"tooth, t = {t:g}")
ax.set_xlabel("depth y below the top of the conductor body [mm]")
ax.set_ylabel("width [mm]")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()

---

## Tests

In [ ]:
#| hide
from dataclasses import replace
from emachines.winding.pattern import slot_positions

In [ ]:
#| hide
# taper: t = 1 gives a constant tooth width, t = 0 a constant slot width
for t in (0.0, 0.3, 1.0):
    st = replace(stator, slot=replace(stator.slot, t=t))
    yy = np.linspace(0, st.slot.hs, 7)
    if t == 1.0:
        assert np.allclose(st.tooth_width_at(yy), st.tooth_width)
    if t == 0.0:
        assert np.allclose(st.slot_width(yy), st.slot.bs)
    # area is the integral of the width
    assert np.isclose(st.slot.area(36), np.trapezoid(st.slot_width(np.linspace(0, st.slot.hs, 2001)),
                                                      np.linspace(0, st.slot.hs, 2001)) if hasattr(np, "trapezoid")
                      else np.trapz(st.slot_width(np.linspace(0, st.slot.hs, 2001)), np.linspace(0, st.slot.hs, 2001)))

# over-under regions: names, order, equal copper area, separators, coverage
for t in (0.0, 0.5, 1.0):
    for h_sep in (0.0, 0.6e-3):
        for n in (2, 3, 6):
            sl = SlotGeometry.semi_closed(2.5e-3, 1e-3, 1.5e-3, 6e-3, 18e-3, t=t, h_sep=h_sep)
            regs = sl.regions("over_under", 36, n)
            assert tuple(r.name for r in regs) == slot_positions("over_under", n)
            areas = [sl.area(36, r.y0, r.y1) for r in regs]
            assert np.allclose(areas, areas[0], rtol=1e-9), (t, h_sep, n, areas)
            assert regs[0].y0 == 0 and np.isclose(regs[-1].y1, 18e-3)
            for a, b in zip(regs, regs[1:]):
                assert np.isclose(b.y0 - a.y1, h_sep)
            # a widening slot makes the deeper layers thinner
            if t > 0:
                assert regs[0].y1 - regs[0].y0 > regs[-1].y1 - regs[-1].y0

# side-by-side and single
lr = SlotGeometry.open(6e-3, 18e-3).regions("side_by_side", 36)
assert [(r.name, r.x0, r.x1, r.y0, r.y1) for r in lr] == [("left", 0.0, 0.5, 0.0, 18e-3), ("right", 0.5, 1.0, 0.0, 18e-3)]
(full,) = SlotGeometry.open(6e-3, 18e-3).regions("single", 36)
assert full.name == "full" and full.y1 == 18e-3

# derived stator values
assert np.isclose(stator.r_body, 62.5e-3) and np.isclose(stator.h_slot, 20.5e-3)
assert np.isclose(stator.tooth_width, 2 * np.pi * 62.5e-3 / 36 - 6e-3)
assert not stator.slot.is_open and SlotGeometry.open(6e-3, 18e-3).is_open

# validation
for f in [lambda: SlotGeometry.open(0, 1e-3), lambda: SlotGeometry.open(6e-3, 18e-3, t=1.2),
          lambda: SlotGeometry.semi_closed(7e-3, 1e-3, 1e-3, 6e-3, 18e-3),
          lambda: SlotGeometry(bs=6e-3, hs=18e-3, b0=3e-3), lambda: SlotGeometry(bs=6e-3, hs=18e-3, b0=-1e-3),
          lambda: SlotGeometry.open(6e-3, 1e-3, h_sep=2e-3).regions("over_under", 36),
          lambda: replace(stator, g=0.0), lambda: replace(stator, Q=200),
          lambda: replace(stator, slot=SlotGeometry.open(11e-3, 18e-3))]:
    try:
        f(); raise AssertionError("expected ValueError")
    except ValueError:
        pass

## References

- Pyrhönen, J., Jokinen, T., Hrabovcová, V. (2014). *Design of Rotating Electrical Machines* (2nd ed.). Wiley — slot dimensions and parallel-tooth slots.
- Lipo, T.A. (2017). *Introduction to AC Machine Design*. Wiley-IEEE Press — slot and tooth proportions.